# Task 2 HardRouter Evaluation (Oracle vs Predicted Routing)
Evaluates the complete Task 2 system combining the trained CorruptionClassifier and 3 independent specialists (salt, blur, occlusion) wrapped in HardRouter. Evaluates on the official validation split, comparing Oracle routing against Predicted routing to isolate the cost of misclassification on restoration quality. Official test set is never touched.


In [ ]:
import torch
print(torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
assert (repo / 'evaluation/evaluate_task2.py').is_file(), 'Push implementation first'
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
os.chdir(repo)
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)


## Run HardRouter Evaluation
Ensure best.pt exists for classifier and all three specialists in Google Drive. Evaluates 2,944 validation cases across all four conditions and severities.


In [ ]:
drive_base = Path('/content/drive/MyDrive/GenAI_A1')
clf_ckpt = drive_base / 'task2-classifier/best.pt'
salt_ckpt = drive_base / 'task2-specialists/salt/best.pt'
blur_ckpt = drive_base / 'task2-specialists/blur/best.pt'
occ_ckpt = drive_base / 'task2-specialists/occlusion/best.pt'

for p in [clf_ckpt, salt_ckpt, blur_ckpt, occ_ckpt]:
    assert p.is_file(), f'Missing checkpoint: {p}'

output_dir = drive_base / 'task2-evaluation-results'
cmd = [
    sys.executable, '-m', 'evaluation.evaluate_task2',
    '--classifier-checkpoint', str(clf_ckpt),
    '--salt-checkpoint', str(salt_ckpt),
    '--blur-checkpoint', str(blur_ckpt),
    '--occlusion-checkpoint', str(occ_ckpt),
    '--output-dir', str(output_dir),
    '--device', 'cuda',
    '--batch-size', '32'
]
subprocess.run(cmd, check=True)


In [ ]:
from IPython.display import display, Image, Markdown
readme = (output_dir / 'README.md').read_text()
display(Markdown(readme))

print('Representative grid:')
display(Image(filename=str(output_dir / 'representative_grid.png')))

print('Routing failure modes:')
display(Image(filename=str(output_dir / 'routing_failures.png')))
